# sparse_release_drawdown_target_tightening_v1

Notebook-first strict target screen after `sparse_release_drawdown_control_bc_ppo_screen_v1` failed. The previous BC handoff kept relative excess above +30pp but missed the <=15% MaxDD target, and a short PPO update worsened `wf_2022` drawdown. This notebook does **not** train or promote a runtime actor. It asks a narrower target-design question: if the same fold-local downtrend states are relabeled with stricter `risk_target_position` caps, is there a non-leaking BC teacher that can meet +30pp relative return and <=15% MaxDD on held-out walk-forward folds?

Governance fields:

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1`.
- **Direct comparison baseline**: `sparse_release_drawdown_control_bc_ppo_screen_v1`.
- **Role**: `diagnose`.
- **Changed layer**: `model_selection` / training target design only.
- **Target windows**: `wf_2022` and `wf_2023_2024`; `wf_2020` remains skipped because the selected target rows include 2020 dates.
- **Accepted trade-off**: lower exposure may be accepted for this drawdown-control screen, but low participation is flagged as a separate risk before PPO or promotion.
- **Failure mode**: no cap reaches +30pp / <=15% MaxDD, or the only passing caps collapse into near-cash behavior that needs a participation objective before PPO.

External basis retained for the design:

- Qlib experiment/replay framework: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- PPO caution: clipped policy optimization can drift from a useful initialization, so short PPO is a diagnostic, not promotion evidence: https://arxiv.org/abs/1707.06347
- Learning from demonstrations / BC warm start before RL: https://arxiv.org/abs/1704.03732
- Backtest overfitting control: https://doi.org/10.21314/JCF.2016.322


In [1]:

from __future__ import annotations

import copy
import json
import random
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
    summarize_bc_slice_fit,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.policy.evaluation import evaluate_split
from src.soxl_quant.policy.positioning import action_levels_for_profile
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'sparse_release_drawdown_target_tightening_v1'
CARD = ROOT / 'runtime/model_selection/card' / EXPERIMENT
SOURCE_CARD = ROOT / 'runtime/model_selection/card/sparse_release_drawdown_control_foldlocal_bc_warmstart_v1'
PPO_SCREEN_CARD = ROOT / 'runtime/model_selection/card/sparse_release_drawdown_control_bc_ppo_screen_v1'
TARGETS_PATH = SOURCE_CARD / 'selected_foldlocal_targets.csv'
FRAME_PATH = ROOT / 'data/soxl_profile_balanced_rule_residual_v22.pkl'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SEED = 20260526
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
MIN_DOWNtrend_SLICE_ACCURACY = 0.60
MIN_DOWNtrend_SAMPLES = 100
CAP_GRID = [0.05, 0.10, 0.15, 0.20, 0.25]
CARD.mkdir(parents=True, exist_ok=True)

WALK_FORWARD_WINDOWS = {
    'wf_2022': {
        'source_target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
        'target_policy': 'native_fold_local',
    },
    'wf_2023_2024': {
        'source_target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
        'target_policy': 'carry_forward_non_overlapping_train_targets',
    },
}


def json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    if pd.isna(value):
        return None
    raise TypeError(f'Object of type {type(value).__name__} is not JSON serializable')


def to_json_safe(value):
    if isinstance(value, dict):
        return {str(k): to_json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_json_safe(v) for v in value]
    if isinstance(value, tuple):
        return [to_json_safe(v) for v in value]
    if isinstance(value, (float, np.floating)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict) -> None:
    path.write_text(
        json.dumps(to_json_safe(payload), indent=2, sort_keys=True, default=json_default, allow_nan=False) + '\n',
        encoding='utf-8',
    )


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:

profile = load_profile(PROFILE_NAME)
source_summary = json.loads((SOURCE_CARD / 'summary.json').read_text(encoding='utf-8'))
source_selection = json.loads((SOURCE_CARD / 'selection.json').read_text(encoding='utf-8'))
ppo_screen_summary = json.loads((PPO_SCREEN_CARD / 'summary.json').read_text(encoding='utf-8'))
ppo_screen_selection = json.loads((PPO_SCREEN_CARD / 'selection.json').read_text(encoding='utf-8'))
base_targets = pd.read_csv(TARGETS_PATH, parse_dates=['datetime'])
frame = pd.read_pickle(FRAME_PATH)

coverage_rows = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    train_start = pd.Timestamp(spec['train'][0])
    train_end = pd.Timestamp(spec['train'][1])
    test_start = pd.Timestamp(spec['test'][0])
    test_end = pd.Timestamp(spec['test'][1])
    fold_targets = base_targets[
        (base_targets['fold'] == spec['source_target_fold'])
        & (base_targets['datetime'] >= train_start)
        & (base_targets['datetime'] <= train_end)
    ].copy()
    eval_overlap = fold_targets[(fold_targets['datetime'] >= test_start) & (fold_targets['datetime'] <= test_end)]
    coverage_rows.append({
        'fold': fold_name,
        'source_target_fold': spec['source_target_fold'],
        'target_policy': spec['target_policy'],
        'target_rows': int(len(fold_targets)),
        'target_start': str(fold_targets['datetime'].min().date()) if len(fold_targets) else '',
        'target_end': str(fold_targets['datetime'].max().date()) if len(fold_targets) else '',
        'train_start': spec['train'][0],
        'train_end': spec['train'][1],
        'valid_start': spec['valid'][0],
        'valid_end': spec['valid'][1],
        'test_start': spec['test'][0],
        'test_end': spec['test'][1],
        'eval_overlap_rows': int(len(eval_overlap)),
        'fold_local_gate': bool(len(fold_targets) > 0 and len(eval_overlap) == 0),
    })
coverage = {
    'experiment': EXPERIMENT,
    'profile': PROFILE_NAME,
    'source_experiment': 'sparse_release_drawdown_control_foldlocal_bc_warmstart_v1',
    'source_decision': source_selection.get('decision'),
    'direct_comparison_experiment': 'sparse_release_drawdown_control_bc_ppo_screen_v1',
    'direct_comparison_decision': ppo_screen_selection.get('decision'),
    'targets_path': str(TARGETS_PATH.relative_to(ROOT)),
    'feature_frame_path': str(FRAME_PATH.relative_to(ROOT)),
    'cap_grid': CAP_GRID,
    'windows': coverage_rows,
    'skipped_windows': [
        {'fold': 'wf_2020', 'reason': 'selected target rows include 2020 dates and would overlap wf_2020 test'}
    ],
}
pd.DataFrame(coverage_rows).to_csv(CARD / 'coverage.csv', index=False)
write_json(CARD / 'coverage.json', coverage)
coverage


{'experiment': 'sparse_release_drawdown_target_tightening_v1',
 'profile': 'sparse_release_bc_ppo_screen_v1',
 'source_experiment': 'sparse_release_drawdown_control_foldlocal_bc_warmstart_v1',
 'source_decision': 'foldlocal_bc_warmstart_ready_no_promotion',
 'direct_comparison_experiment': 'sparse_release_drawdown_control_bc_ppo_screen_v1',
 'direct_comparison_decision': 'drawdown_control_bc_ppo_screen_failed_no_promotion',
 'targets_path': 'runtime/model_selection/card/sparse_release_drawdown_control_foldlocal_bc_warmstart_v1/selected_foldlocal_targets.csv',
 'feature_frame_path': 'data/soxl_profile_balanced_rule_residual_v22.pkl',
 'cap_grid': [0.05, 0.1, 0.15, 0.2, 0.25],
 'windows': [{'fold': 'wf_2022',
   'source_target_fold': 'wf_2022_train',
   'target_policy': 'native_fold_local',
   'target_rows': 370,
   'target_start': '2011-06-28',
   'target_end': '2020-05-28',
   'train_start': '2010-03-11',
   'train_end': '2020-12-31',
   'valid_start': '2021-01-01',
   'valid_end': '20

In [3]:

def evaluate_actor_path(actor, eval_df: pd.DataFrame, price_series: pd.Series, profile, test_seg: tuple[str, str]) -> dict[str, float]:
    feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
    split_metrics = evaluate_split(
        actor,
        eval_df,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        asset_idx=0,
        action_levels=action_levels_for_profile(profile),
        observation_window=profile.model.observation_window,
        hard_floor=profile.model.hard_floor,
        use_hybrid_overlay=profile.model.use_hybrid_overlay,
        delta_levels=np.asarray(profile.model.delta_levels, dtype=np.float32) if profile.model.use_hybrid_overlay else None,
        feature_columns=feature_columns,
        mode_router_config=profile.mode_router,
        position_mapping=profile.model.position_mapping,
    )
    bt = backtest_profile(
        actor,
        eval_df,
        price_series,
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        asset_idx=0,
        feature_columns=feature_columns,
        model_config=profile.model,
        mode_router_config=profile.mode_router,
        test_seg=test_seg,
    )
    out = dict(split_metrics)
    for src, dst in [
        ('return', 'return'),
        ('sharpe', 'sharpe'),
        ('max_dd', 'max_dd'),
        ('bnh_return', 'buy_and_hold_return'),
        ('relative_outperformance', 'relative_return'),
    ]:
        if src in bt:
            out[dst] = float(bt[src])
    out['backtest_return'] = float(bt.get('return', out.get('return', 0.0)))
    out['backtest_max_dd'] = float(bt.get('max_dd', out.get('max_dd', 0.0)))
    out['backtest_relative_return'] = float(bt.get('relative_outperformance', out.get('relative_return', 0.0)))
    return out


def prefixed_metrics(prefix: str, metrics: dict) -> dict[str, float]:
    keys = ['return', 'sharpe', 'max_dd', 'avg_pos', 'stress_avg_pos', 'buy_and_hold_return', 'relative_return']
    return {f'{prefix}_{k}': float(metrics[k]) for k in keys if k in metrics}


def target_score(relative_return: float, max_dd: float, avg_pos: float) -> float:
    relative_gap = max(0.0, TARGET_RELATIVE_RETURN - float(relative_return))
    drawdown_gap = max(0.0, float(max_dd) - TARGET_MAX_DD)
    near_cash_penalty = max(0.0, 0.10 - float(avg_pos)) * 20.0
    return relative_gap + drawdown_gap * 4.0 + near_cash_penalty


In [4]:

init_qlib(QLIB_DATA_PATH)
train_config = StateObjectiveBCTrainingConfig(steps=200, lr=0.001, batch_size=128, seed=SEED)
weight_config = StateObjectiveBCSampleWeightConfig(
    downtrend_multiplier=2.0,
    rebound_multiplier=1.0,
    ordinary_multiplier=1.0,
    objective_weight_power=0.5,
    class_balanced_beta=0.99,
    weight_cap=20.0,
)

fold_datasets = {}
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    price_frame = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index()
    fold_datasets[fold_name] = {
        'dataset': dataset,
        'test_df': segment_frame(dataset, 'test'),
        'test_price': price_frame.loc[spec['test'][0]:spec['test'][1]],
    }

candidate_rows = []
fold_rows = []
slice_frames = []
action_frames = []
selected_target_frames = []

for cap in CAP_GRID:
    set_seed(SEED + int(round(cap * 1000)))
    cap_targets = base_targets.copy()
    cap_targets['risk_target_position'] = float(cap)
    cap_targets['target_cap_variant'] = f'cap_{cap:.2f}'
    selected_target_frames.append(cap_targets)

    bc_result = fit_state_objective_bc_actor_from_frame(
        frame,
        targets=cap_targets,
        profile=profile,
        train_config=train_config,
        sample_weight_config=weight_config,
        sample_size=0,
    )
    bc_dataset = build_state_objective_bc_dataset_from_frame(
        frame,
        targets=cap_targets,
        profile=profile,
        sample_size=0,
    )
    slice_metrics = evaluate_bc_actor_slice_fit(
        bc_result.actor,
        bc_result.obs,
        bc_result.actions,
        bc_dataset.metadata,
        slice_cols=('is_downtrend_state', 'is_rebound_cluster'),
    )
    slice_summary = summarize_bc_slice_fit(
        slice_metrics,
        required_slices=('is_downtrend_state',),
        min_accuracy=MIN_DOWNtrend_SLICE_ACCURACY,
        min_samples=MIN_DOWNtrend_SAMPLES,
    )
    tagged_slices = slice_metrics.copy()
    tagged_slices.insert(0, 'target_cap', cap)
    slice_frames.append(tagged_slices)
    action_dist = bc_result.action_distribution.copy()
    action_dist.insert(0, 'target_cap', cap)
    action_frames.append(action_dist)

    cap_fold_rows = []
    actor = bc_result.actor
    actor.eval()
    for fold_name, spec in WALK_FORWARD_WINDOWS.items():
        data = fold_datasets[fold_name]
        metrics = evaluate_actor_path(actor, data['test_df'], data['test_price'], profile, spec['test'])
        gate = bool(metrics.get('relative_return', -1e9) >= TARGET_RELATIVE_RETURN and metrics.get('max_dd', 1e9) <= TARGET_MAX_DD)
        row = {
            'target_cap': cap,
            'fold': fold_name,
            'status': 'evaluated',
            'target_rows': int(len(cap_targets)),
            'bc_after_accuracy': float(bc_result.metrics['after_accuracy']),
            'bc_after_loss': float(bc_result.metrics['after_loss']),
            'bc_weight_max': float(bc_result.metrics.get('weight_max', float('nan'))),
            'bc_slice_gate': bool(slice_summary['slice_fit_gate']),
            'fold_gate': gate,
            'target_score': target_score(
                float(metrics.get('relative_return', 0.0)),
                float(metrics.get('max_dd', 0.0)),
                float(metrics.get('avg_pos', 0.0)),
            ),
            **prefixed_metrics('test', metrics),
        }
        fold_rows.append(row)
        cap_fold_rows.append(row)
    cap_frame = pd.DataFrame(cap_fold_rows)
    candidate_rows.append({
        'target_cap': cap,
        'folds_evaluated': int(len(cap_frame)),
        'fold_gate_pass_count': int(cap_frame['fold_gate'].sum()),
        'all_fold_gate': bool(cap_frame['fold_gate'].all()),
        'mean_target_score': float(cap_frame['target_score'].mean()),
        'worst_target_score': float(cap_frame['target_score'].max()),
        'mean_relative_return': float(cap_frame['test_relative_return'].mean()),
        'worst_max_dd': float(cap_frame['test_max_dd'].max()),
        'mean_avg_pos': float(cap_frame['test_avg_pos'].mean()),
        'bc_after_accuracy': float(bc_result.metrics['after_accuracy']),
        'slice_gate': bool(slice_summary['slice_fit_gate']),
    })

candidate_summary = pd.DataFrame(candidate_rows).sort_values(['all_fold_gate', 'worst_target_score', 'mean_target_score'], ascending=[False, True, True]).reset_index(drop=True)
fold_results = pd.DataFrame(fold_rows)
bc_slice_metrics = pd.concat(slice_frames, ignore_index=True) if slice_frames else pd.DataFrame()
action_distribution = pd.concat(action_frames, ignore_index=True) if action_frames else pd.DataFrame()
selected_targets_by_cap = pd.concat(selected_target_frames, ignore_index=True) if selected_target_frames else pd.DataFrame()

candidate_summary.to_csv(CARD / 'target_cap_summary.csv', index=False)
fold_results.to_csv(CARD / 'target_cap_fold_results.csv', index=False)
bc_slice_metrics.to_csv(CARD / 'bc_slice_metrics.csv', index=False)
action_distribution.to_csv(CARD / 'action_distribution.csv', index=False)
selected_targets_by_cap.to_csv(CARD / 'selected_targets_by_cap.csv', index=False)

display(candidate_summary)
display(fold_results)


[72230:MainThread](2026-05-26 05:10:26,592) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[72230:MainThread](2026-05-26 05:10:27,257) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[72230:MainThread](2026-05-26 05:10:27,258) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[72230:MainThread](2026-05-26 05:10:29,667) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[72230:MainThread](2026-05-26 05:10:29,670) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

,target_cap,folds_evaluated,fold_gate_pass_count,all_fold_gate,mean_target_score,worst_target_score,mean_relative_return,worst_max_dd,mean_avg_pos,bc_after_accuracy,slice_gate
0,0.10,2,2,True,0.000000,0.000000,54.941481,9.875679,0.117553,0.937838,True
1,0.05,2,2,True,0.261215,0.284211,56.444637,9.853415,0.086939,0.927027,True
2,0.15,2,1,False,6.907359,13.814717,51.480658,18.453679,0.149651,0.929730,True
3,0.20,2,0,False,16.069602,21.677030,46.187406,20.419257,0.202473,0.927027,True
4,0.25,2,0,False,41.651707,57.462531,47.748472,29.365633,0.251918,0.916216,True


,target_cap,fold,status,target_rows,bc_after_accuracy,bc_after_loss,bc_weight_max,bc_slice_gate,fold_gate,target_score,test_return,test_sharpe,test_max_dd,test_avg_pos,test_stress_avg_pos,test_buy_and_hold_return,test_relative_return
0,0.05,wf_2022,evaluated,370,0.927027,0.980755,1.591958,True,True,0.284211,-5.464802,-0.487804,9.853415,0.085789,0.053779,-76.614257,75.101201
1,0.05,wf_2023_2024,evaluated,370,0.927027,0.980755,1.591958,True,True,0.238220,-1.266854,-0.129580,8.204373,0.088089,0.056338,-39.952710,37.788073
2,0.10,wf_2022,evaluated,370,0.937838,0.971698,1.591958,True,True,0.000000,-7.000281,-0.657429,9.875679,0.105263,0.094767,-76.614257,72.322346
3,0.10,wf_2023_2024,evaluated,370,0.937838,0.971698,1.591958,True,True,0.000000,-2.958779,-0.370753,9.737972,0.129843,0.113028,-39.952710,37.560616
4,0.15,wf_2022,evaluated,370,0.929730,0.978051,6.232017,True,False,13.814717,-14.262698,-0.825246,18.453679,0.141842,0.102471,-76.614257,65.767710
5,0.15,wf_2023_2024,evaluated,370,0.929730,0.978051,6.232017,True,True,0.000000,-7.941327,-0.601063,13.359741,0.157461,0.118310,-39.952710,37.193607
6,0.20,wf_2022,evaluated,370,0.927027,0.984345,6.378792,True,False,21.677030,-16.621878,-0.973853,20.419257,0.206189,0.173401,-76.614257,58.021225
7,0.20,wf_2023_2024,evaluated,370,0.927027,0.984345,6.378792,True,False,10.462173,-12.890973,-0.766277,17.615543,0.198757,0.184331,-39.952710,34.353586
8,0.25,wf_2022,evaluated,370,0.916216,0.993203,6.115695,True,False,57.462531,-24.634159,-1.024182,29.365633,0.269413,0.246221,-76.614257,54.403539
9,0.25,wf_2023_2024,evaluated,370,0.916216,0.993203,6.115695,True,False,25.840883,-11.480984,-0.612625,21.460221,0.234424,0.247183,-39.952710,41.093405


In [5]:

evaluated = fold_results[fold_results['status'].eq('evaluated')].copy() if not fold_results.empty else pd.DataFrame()
passing_caps = candidate_summary[candidate_summary['all_fold_gate'].astype(bool)].copy() if not candidate_summary.empty else pd.DataFrame()
if not passing_caps.empty:
    selected = passing_caps.sort_values(['mean_avg_pos', 'worst_max_dd'], ascending=[False, True]).iloc[0].to_dict()
    decision = 'drawdown_target_tightening_passed_partial_wf_no_promotion'
    next_mechanism = 'train_state_loss_with_selected_cap_then_full_fixed_split_walkforward'
else:
    selected = candidate_summary.iloc[0].to_dict() if not candidate_summary.empty else {}
    decision = 'drawdown_target_tightening_failed_no_promotion'
    next_mechanism = 'add_sequence_risk_feature_or_stronger_drawdown_loss_before_bc_or_ppo'
selected_cap = float(selected.get('target_cap', float('nan'))) if selected else float('nan')
near_cash_warning = bool(np.isfinite(selected_cap) and float(selected.get('mean_avg_pos', 0.0)) < 0.12)

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'sparse_release_drawdown_control_bc_ppo_screen_v1',
    'role': 'diagnose',
    'profile': PROFILE_NAME,
    'changed_layer': ['model_selection', 'training_target_design'],
    'promotion_allowed': False,
    'promotion_blocker': 'This is a target-design BC screen only; no actor bundle is promoted or retained.',
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'cap_grid': CAP_GRID,
    'selected_target_cap': selected_cap if np.isfinite(selected_cap) else None,
    'passing_cap_count': int(len(passing_caps)),
    'selected_near_cash_warning': near_cash_warning,
    'folds_evaluated': sorted(evaluated['fold'].unique().tolist()) if not evaluated.empty else [],
    'sidecars': [
        'coverage.csv',
        'coverage.json',
        'target_cap_summary.csv',
        'target_cap_fold_results.csv',
        'bc_slice_metrics.csv',
        'action_distribution.csv',
        'selected_targets_by_cap.csv',
        'selection.json',
        'summary.json',
    ],
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://arxiv.org/abs/1707.06347',
        'https://arxiv.org/abs/1704.03732',
        'https://doi.org/10.21314/JCF.2016.322',
    ],
}
summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'selection': selection,
    'coverage': coverage,
    'selected_candidate': selected,
    'candidate_summary': candidate_summary.to_dict(orient='records'),
    'fold_results': fold_results.to_dict(orient='records'),
    'direct_comparison': {
        'experiment': 'sparse_release_drawdown_control_bc_ppo_screen_v1',
        'decision': ppo_screen_selection.get('decision'),
        'conclusion': ppo_screen_summary.get('conclusion', []),
    },
    'conclusion': [],
}
if candidate_summary.empty:
    summary['conclusion'].append('No target caps were evaluated; do not continue this branch.')
else:
    for row in candidate_summary.sort_values('target_cap').to_dict(orient='records'):
        summary['conclusion'].append(
            f"cap={row['target_cap']:.2f}: pass={bool(row['all_fold_gate'])}, pass_count={int(row['fold_gate_pass_count'])}/2, "
            f"worst MaxDD={row['worst_max_dd']:.2f}%, mean relative={row['mean_relative_return']:.2f}pp, mean avg_pos={row['mean_avg_pos']:.3f}."
        )
    if not passing_caps.empty:
        summary['conclusion'].append(
            f"Selected diagnostic cap={selected_cap:.2f}: it passes the two evaluated folds but is not a promotion because wf_2020/full fixed split are not covered and low participation must be checked."
        )
    else:
        summary['conclusion'].append(
            'No cap in the sweep passes both evaluated folds; target tightening alone is insufficient.'
        )
    if near_cash_warning:
        summary['conclusion'].append(
            'The selected cap is close to a cash-defense solution; next step must add a participation/rebound objective before PPO, not promote this actor.'
        )

write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
summary


{'experiment': 'sparse_release_drawdown_target_tightening_v1',
 'decision': 'drawdown_target_tightening_passed_partial_wf_no_promotion',
 'selection': {'experiment': 'sparse_release_drawdown_target_tightening_v1',
  'created_at': '2026-05-25T21:11:43.013563+00:00',
  'decision': 'drawdown_target_tightening_passed_partial_wf_no_promotion',
  'next_mechanism': 'train_state_loss_with_selected_cap_then_full_fixed_split_walkforward',
  'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
  'direct_comparison_baseline': 'sparse_release_drawdown_control_bc_ppo_screen_v1',
  'role': 'diagnose',
  'profile': 'sparse_release_bc_ppo_screen_v1',
  'changed_layer': ['model_selection', 'training_target_design'],
  'promotion_allowed': False,
  'promotion_blocker': 'This is a target-design BC screen only; no actor bundle is promoted or retained.',
  'target_relative_return_pp': 30.0,
  'target_max_dd_pct': 15.0,
  'cap_grid': [0.05, 0.1, 0.15, 0.2, 0.25],
  'selected_target_cap': 0.1,
  'pa

In [6]:

lines = [
    f"## Runtime Conclusion: {EXPERIMENT}",
    "",
    f"- **Decision**: `{summary['decision']}`.",
    f"- **Promotion**: blocked. {selection['promotion_blocker']}",
    f"- **Selected diagnostic cap**: `{selection['selected_target_cap']}`.",
    f"- **Passing caps**: {selection['passing_cap_count']} / {len(CAP_GRID)}.",
    f"- **Near-cash warning**: `{selection['selected_near_cash_warning']}`.",
    "",
    "### Cap Sweep Evidence",
]
for row in candidate_summary.sort_values('target_cap').to_dict(orient='records'):
    lines.append(
        f"- `cap={row['target_cap']:.2f}`: pass={bool(row['all_fold_gate'])}, "
        f"pass_count={int(row['fold_gate_pass_count'])}/2, worst MaxDD={row['worst_max_dd']:.2f}%, "
        f"mean relative={row['mean_relative_return']:.2f}pp, mean avg_pos={row['mean_avg_pos']:.3f}."
    )
lines += [
    "",
    "### Risk Notes",
    "- This is a BC target-design screen, not a full training branch.",
    "- `wf_2020` is intentionally skipped because selected target rows include 2020 dates.",
    "- A passing low cap may still be an underparticipation solution; require fixed-split, wf_2020, and rebound/trend participation checks before PPO or promotion.",
    "",
    "### Artifacts",
]
for sidecar in selection['sidecars']:
    lines.append(f"- `{sidecar}`")
lines += [
    "",
    "### Next Action",
    f"- `{selection['next_mechanism']}`.",
]
display(Markdown("\n".join(lines)))


## Runtime Conclusion: sparse_release_drawdown_target_tightening_v1

- **Decision**: `drawdown_target_tightening_passed_partial_wf_no_promotion`.
- **Promotion**: blocked. This is a target-design BC screen only; no actor bundle is promoted or retained.
- **Selected diagnostic cap**: `0.1`.
- **Passing caps**: 2 / 5.
- **Near-cash warning**: `True`.

### Cap Sweep Evidence
- `cap=0.05`: pass=True, pass_count=2/2, worst MaxDD=9.85%, mean relative=56.44pp, mean avg_pos=0.087.
- `cap=0.10`: pass=True, pass_count=2/2, worst MaxDD=9.88%, mean relative=54.94pp, mean avg_pos=0.118.
- `cap=0.15`: pass=False, pass_count=1/2, worst MaxDD=18.45%, mean relative=51.48pp, mean avg_pos=0.150.
- `cap=0.20`: pass=False, pass_count=0/2, worst MaxDD=20.42%, mean relative=46.19pp, mean avg_pos=0.202.
- `cap=0.25`: pass=False, pass_count=0/2, worst MaxDD=29.37%, mean relative=47.75pp, mean avg_pos=0.252.

### Risk Notes
- This is a BC target-design screen, not a full training branch.
- `wf_2020` is intentionally skipped because selected target rows include 2020 dates.
- A passing low cap may still be an underparticipation solution; require fixed-split, wf_2020, and rebound/trend participation checks before PPO or promotion.

### Artifacts
- `coverage.csv`
- `coverage.json`
- `target_cap_summary.csv`
- `target_cap_fold_results.csv`
- `bc_slice_metrics.csv`
- `action_distribution.csv`
- `selected_targets_by_cap.csv`
- `selection.json`
- `summary.json`

### Next Action
- `train_state_loss_with_selected_cap_then_full_fixed_split_walkforward`.